# MIE 402 — Pre-Lab 1: Sampling a Two-Tone Signal

**Fall 2026 — Dynamic Data Sampling and Frequency Analysis**

This pre-lab prepares you to recognize adequate sampling, aliasing, and FFT peaks before you collect sound data with the Moku:Go. All Python code is provided. Your work is to calculate, predict, run, compare, and explain the results.

## Submission

- Run every code cell and keep all figures visible.
- Type short answers in the response cells.
- Export the completed notebook to HTML or PDF and submit it on Canvas **before your own laboratory section begins**.
- The pre-lab is normally posted on Canvas on the Monday before the laboratory. Late pre-labs are not accepted.

You may discuss general approaches, but your submitted notebook and explanations must be your own work.


## Theory review

### Continuous and sampled signals

A physical voltage or sound signal changes continuously with time. A data-acquisition system stores its value only at selected times. If the sample rate is $f_s$ samples per second, the time between samples is

$$\Delta t=\frac{1}{f_s}.$$

For a record of duration $T$, the number of stored samples is approximately $N=f_sT$.

### Mean and RMS

The mean describes the average signed level of a record:

$$\bar{x}=\frac{1}{N}\sum_{n=0}^{N-1}x_n.$$

The root-mean-square value describes the overall size of an oscillating signal:

$$x_{RMS}=\sqrt{\frac{1}{N}\sum_{n=0}^{N-1}x_n^2}.$$

A zero-mean sinusoid with amplitude $A$ has $x_{RMS}=A/\sqrt{2}$. For distinct sinusoidal components measured over complete cycles, their mean-square contributions add.

### Nyquist frequency and aliasing

The highest frequency that a sampled record can represent without ambiguity is the Nyquist frequency:

$$f_N=\frac{f_s}{2}.$$

A component above $f_N$ appears at a false lower frequency. This effect is **aliasing**. A useful calculation is

$$f_{alias}=|f-kf_s|,$$

where integer $k$ places $f_{alias}$ between 0 and $f_s/2$. Once aliasing occurs, the stored samples alone cannot recover the original frequency.

### FFT and frequency resolution

The Fast Fourier Transform converts a sampled time record into frequency components. Peaks in the one-sided magnitude spectrum indicate the frequencies present in the record. The spacing between FFT frequency bins is approximately

$$\Delta f=\frac{f_s}{N}=\frac{1}{T}.$$

A longer record gives smaller $\Delta f$ and separates nearby frequencies more clearly. A higher sample rate increases the measurable frequency range. These two settings solve different problems.

Read this section before running the analysis. In the questions below, use these relationships to predict the results before viewing the FFT output.


## New problem

Consider the two-tone voltage signal

$$x(t)=1.20\sin(2\pi(18)t+25^\circ)+0.45\cos(2\pi(42)t-15^\circ)\quad\text{V}.$$

The 18 Hz component represents the desired signal and the 42 Hz component represents a second tone that could be present in a measurement.

1. Calculate the theoretical mean and RMS over a complete one-second record.
2. Run the sampling analysis for **336, 126, 72, and 48 Hz** and compare the sampled mean and RMS.
3. Predict the two FFT peaks for every case before running the FFT.
4. Run the one-sided FFT analysis and compare the observed peaks with your predictions.
5. Compare 0.25 s and 1.00 s records and explain the change in frequency-bin spacing.
6. Design and justify a sample rate and record duration for a new measurement containing frequencies up to 75 Hz.

These values and the two-tone signal differ from the Spring 2026 pre-lab while practicing the same concepts needed in Lab 1. You do not need to write or complete Python code. Total: **20 points**.


## 1. Define the signal and analysis functions

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

plt.rcParams.update({"figure.figsize": (9, 4.8), "font.size": 11})

# Signal definition
A1, f1, phi1_deg = 1.20, 18.0, 25.0
A2, f2, phi2_deg = 0.45, 42.0, -15.0
duration = 1.00
sample_rates = [336.0, 126.0, 72.0, 48.0]

def signal(t):
    phi1 = np.deg2rad(phi1_deg)
    phi2 = np.deg2rad(phi2_deg)
    return (A1*np.sin(2*np.pi*f1*t + phi1)
            + A2*np.cos(2*np.pi*f2*t + phi2))

def mean_and_rms(x):
    return np.mean(x), np.sqrt(np.mean(x**2))

def one_sided_spectrum(x, fs):
    n = len(x)
    window = np.hanning(n)
    coherent_gain = np.mean(window)
    magnitude = np.abs(np.fft.rfft(x*window))/(n*coherent_gain)
    if n > 2:
        magnitude[1:-1] *= 2
    frequency = np.fft.rfftfreq(n, d=1/fs)
    return frequency, magnitude


## 2. Theoretical reference

In [ ]:
# Dense reference waveform and theoretical values
t_ref = np.linspace(0, duration, 20001, endpoint=False)
x_ref = signal(t_ref)

theoretical_mean = 0.0
theoretical_rms = np.sqrt((A1**2 + A2**2)/2)

print(f"Theoretical mean = {theoretical_mean:.4f} V")
print(f"Theoretical RMS  = {theoretical_rms:.4f} V")

plt.plot(t_ref, x_ref)
plt.xlim(0, 0.20)
plt.xlabel("Time (s)")
plt.ylabel("Voltage (V)")
plt.title("Dense reference waveform — first 0.20 s")
plt.grid(True)
plt.show()


## 3. Sampled time histories

In [ ]:
# Sample the signal at four rates and compare it with the reference
records = {}
fig, axes = plt.subplots(2, 2, figsize=(12, 8), sharex=True, sharey=True)

for ax, fs in zip(axes.flat, sample_rates):
    n = int(round(duration*fs))
    t = np.arange(n)/fs
    x = signal(t)
    records[fs] = (t, x)
    sample_mean, sample_rms = mean_and_rms(x)

    ax.plot(t_ref, x_ref, color="0.75", lw=1.2, label="Reference")
    ax.plot(t, x, "o-", ms=3, lw=0.9, label="Samples")
    ax.set_xlim(0, 0.20)
    ax.set_title(f"fs = {fs:g} Hz | mean={sample_mean:.4f} V | RMS={sample_rms:.4f} V")
    ax.set_xlabel("Time (s)")
    ax.set_ylabel("Voltage (V)")
    ax.grid(True)

axes.flat[0].legend()
fig.suptitle("Time-domain comparison", fontsize=14)
fig.tight_layout()
plt.show()


## Task 3 — Predictions before FFT (4 points)

Complete this table **before** running the next code cell. Show the alias calculation when a tone exceeds the Nyquist frequency.

| $f_s$ (Hz) | Nyquist (Hz) | Predicted peak 1 (Hz) | Predicted peak 2 (Hz) | Calculation or reason |
|---:|---:|---:|---:|---|
| 336 |  |  |  |  |
| 126 |  |  |  |  |
| 72  |  |  |  |  |
| 48  |  |  |  |  |


## 4. Frequency-domain analysis

In [ ]:
# Compute and plot the one-sided spectra
fig, axes = plt.subplots(2, 2, figsize=(12, 8), sharex=True, sharey=True)

for ax, fs in zip(axes.flat, sample_rates):
    _, x = records[fs]
    freq, mag = one_sided_spectrum(x, fs)
    ax.stem(freq, mag, basefmt=" ")
    ax.axvline(f1, color="tab:green", ls="--", lw=1, label="18 Hz expected")
    ax.axvline(f2, color="tab:red", ls="--", lw=1, label="42 Hz expected")
    ax.set_xlim(0, 60)
    ax.set_ylim(0, 1.35)
    ax.set_title(f"fs = {fs:g} Hz; Nyquist = {fs/2:g} Hz")
    ax.set_xlabel("Frequency (Hz)")
    ax.set_ylabel("Magnitude (V)")
    ax.grid(True)

axes.flat[0].legend(loc="upper right")
fig.suptitle("One-sided FFT magnitude spectra", fontsize=14)
fig.tight_layout()
plt.show()


In [ ]:
# List the strongest spectral peaks to support your interpretation
for fs in sample_rates:
    _, x = records[fs]
    freq, mag = one_sided_spectrum(x, fs)
    candidate = np.argsort(mag[1:])[-4:] + 1
    candidate = candidate[np.argsort(mag[candidate])[::-1]]
    strongest = [(round(float(freq[i]), 2), round(float(mag[i]), 3)) for i in candidate]
    print(f"fs={fs:6.1f} Hz, Nyquist={fs/2:5.1f} Hz, strongest bins={strongest}")


## 5. Record duration and FFT resolution

In [ ]:
# Task 5 (3 points): frequency resolution
for test_duration in [0.25, 1.00]:
    fs = 126.0
    n = int(round(test_duration*fs))
    t = np.arange(n)/fs
    x = signal(t)
    frequency, magnitude = one_sided_spectrum(x, fs)
    delta_f = fs/n
    print(f"duration={test_duration:.2f} s, N={n}, delta_f={delta_f:.2f} Hz")


## Data-analysis responses

### Task 1: Mean and RMS (3 points)

Show the theoretical mean and RMS calculation. Explain why the mean is zero over the one-second record.

**Response:**  


### Task 2: Time-domain comparison (3 points)

Use the output to complete the table. Calculate RMS percent error as $100|RMS_{sample}-RMS_{theory}|/RMS_{theory}$.

| $f_s$ (Hz) | Samples per 42 Hz cycle | Sampled mean (V) | Sampled RMS (V) | RMS error (%) |
|---:|---:|---:|---:|---:|
| 336 |  |  |  |  |
| 126 |  |  |  |  |
| 72  |  |  |  |  |
| 48  |  |  |  |  |

Which sample rate gives the most convincing time-domain trace, and why?

**Response:**  


### Task 4: FFT interpretation (4 points)

List the two dominant FFT frequencies for every sample rate. Compare them with your predictions and explain each aliased frequency.

**Response:**  


### Task 5: Record duration (2 points)

Explain why the 1.00 s record has finer frequency-bin spacing than the 0.25 s record. State both values of $\Delta f$.

**Response:**  


### Task 6: Measurement design (3 points)

You expect an unknown experimental signal to contain frequencies up to 75 Hz. Choose a sample rate and record duration. Your design must provide at least **five samples per cycle** at 75 Hz and frequency-bin spacing no larger than **0.5 Hz**. Show both checks.

**Response:**  


### Engineering interpretation

In 3–5 sentences, explain why checking only a time history is insufficient when validating dynamic experimental data. Connect your explanation to the Moku:Go sound measurements in Lab 1.

**Response:**  
